In [1]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("spam-experiment")

<Experiment: artifact_location='/mnt/c/Users/user/Documents/Projects/mlops-exer-1/notebooks/mlruns/1', creation_time=1780963326545, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780963326545, lifecycle_stage='active', name='spam-experiment', tags={}, trace_location=None, workspace='default'>

In [2]:
# from spam_checker.data.spam_data_module import SPAMDataModule

In [3]:
# spam_data = SPAMDataModule()

In [4]:
# spam_data.prepare_data()

In [5]:
import pandas as pd

# Paste your copied file path inside the quotes
# df = pd.read_csv('../test_dataset/spam.csv')

df = pd.read_csv('../test_dataset/spam.csv', encoding_errors='ignore')

df = df[['v1', 'v2']]

# df['v1'] = df['v1'].replace('spam', '1')
# df['v1'] = df['v1'].replace('ham', '0')
df = df.rename(columns={"v1": "label", "v2": "text"})

# View the first few rows of your data
df.head(10)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [6]:
df['label'] = df['label'].replace('spam', '1')
df['label'] = df['label'].replace('ham', '0')

# View the first few rows of your data
df.head(10)

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [7]:
len(df.index)

5572

In [8]:
empty_spam_rows = df[df['text'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, text]
Index: []


In [9]:
empty_count = df['text'].isna().sum()
print(f"Empty cells: {empty_count}")

Empty cells: 0


In [10]:
# !python -m spacy download en_core_web_sm

In [11]:
# import spacy

# nlp = spacy.load("en_core_web_sm")

In [12]:
# doc = nlp("spaCy is successfully working in Jupyter!")
# for token in doc:
#     print(token.text, token.pos_)

In [13]:
# # Function to clean the text
# def clean_text(text):
#     doc = nlp(text)  # Process the text with spaCy
#     cleaned_tokens = []
#     for token in doc:
#         # Remove stop words, punctuation, and retain only alphabetic words
#         if not token.is_stop and not token.is_punct and token.is_alpha:
#             cleaned_tokens.append(token.lemma_)  # Append the lemmatized version of the token
#     return " ".join(cleaned_tokens)

# Apply the cleaning function to the 'text' column in the dataset
# df['cleaned_text'] = df['text'].apply(clean_text)

In [14]:
df.head(10)

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [15]:
training_input_labels = df.loc[:, df.columns == 'label']
training_input_text = df.loc[:, df.columns != 'label']

print(training_input_labels['label'].value_counts())

label
0    4825
1     747
Name: count, dtype: int64


In [16]:
print(training_input_labels)

     label
0        0
1        0
2        1
3        0
4        0
...    ...
5567     1
5568     0
5569     0
5570     0
5571     0

[5572 rows x 1 columns]


In [17]:
from sklearn.model_selection import train_test_split

train_text_temp, val_text_final, train_labels_temp, val_labels = train_test_split(
    training_input_text, training_input_labels, train_size=0.5, random_state=0, stratify=training_input_labels)

train_text_final, test_text_final, train_labels, test_labels = train_test_split(
    train_text_temp, train_labels_temp, train_size=0.8, random_state=0, stratify=train_labels_temp)


In [18]:
train_text_combined = pd.concat([train_labels, train_text_final], axis=1).reset_index(drop=True)

In [19]:
test_text_combined = pd.concat([test_labels, test_text_final], axis=1).reset_index(drop=True)

In [20]:
val_text_combined = pd.concat([val_labels, val_text_final], axis=1).reset_index(drop=True)

In [21]:
empty_spam_rows = train_text_combined[train_text_combined['text'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, text]
Index: []


In [22]:
# empty_spam_rows = train_text_combined[train_text_combined['cleaned_text'].isna()]
# print(empty_spam_rows)

In [23]:
empty_spam_rows = train_text_combined[train_text_combined['label'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, text]
Index: []


In [24]:
# def spacy_tokenizer(text):
#     # Use the clean_lemm function first
#     cleaned_text = clean_text(text)
#     # Then tokenize the cleaned text
#     return cleaned_text.split()

In [25]:
# from collections import defaultdict

# def build_vocab(text_iterator, min_freq=3, specials=('<unk>', '<pad>')):
#     token_counts = defaultdict(int)
#     for text in text_iterator:
#         for token in spacy_tokenizer(text):
#             token_counts[token] += 1
#     vocab = {token: idx for idx, (token, count) in enumerate(token_counts.items()) if count >= min_freq}
#     for special in specials:
#         if special not in vocab:
#             vocab[special] = len(vocab) 
#     return vocab
# vocab = build_vocab(df['text'])
# # vocab = build_vocab(df['cleaned_text'])
# vocab_size = len(vocab)

In [26]:
# print(vocab_size)

In [27]:
# print(vocab)

In [28]:
import torch
import torch.utils.data as data
from pytorch_lightning import LightningDataModule
from collections import defaultdict
from collections import Counter
import re

import spacy

nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    doc = nlp(text)  # Process the text with spaCy
    cleaned_tokens = []
    for token in doc:
        # Remove stop words, punctuation, and retain only alphabetic words
        if not token.is_stop and not token.is_punct and token.is_alpha:
            cleaned_tokens.append(token.lemma_)  # Append the lemmatized version of the token
    return " ".join(cleaned_tokens)

def spacy_tokenizer(text):
    # Use the clean_lemm function first
    cleaned_text = clean_text(text)
    # Then tokenize the cleaned text
    return cleaned_text.split()

class SMSDataModule(LightningDataModule):
    # def __init__(self, texts, labels, batch_size=32):
    def __init__(self, data, batch_size=32):
        super().__init__()
        self.texts = data['text']
        self.labels = data['label']
        self.batch_size = batch_size
        self.vocab = None

        # print(self.texts)
        # print(self.labels)
        
    def _yield_tokens(self, data_iter):
        for text in data_iter:
            yield text.split() # Simple space-based tokenization

    def build_vocab(self):
        # text_iterator = self.texts
        # min_freq=2
        # specials=['<unk>', '<pad>']

        # token_counts = defaultdict(int)
        # for text in text_iterator:
        #     for token in spacy_tokenizer(text):
        #         token_counts[token] += 1
        # vocab = {token: idx for idx, (token, count) in enumerate(token_counts.items()) if count >= min_freq}
        # for special in specials:
        #     if special not in vocab:
        #         vocab[special] = len(vocab)

        # self.vocab = vocab
        

        # 2. Text Preprocessing and Vocabulary Builder
        def clean_text(text):
            text = text.lower()
            text = re.sub(r'[^a-z0-9\s]', '', text)
            return text.split()

        # Build Vocabulary
        all_words = []
        for text in self.texts:
            all_words.extend(clean_text(text))

        word_counts = Counter(all_words)
        # Filter words that appear at least once; add <PAD> and <UNK> tokens
        vocab = {"<pad>": 0, "<unk>": 1}
        for word in word_counts:
            if word not in vocab:
                vocab[word] = len(vocab)

        # VOCAB_SIZE = len(vocab)
        # MAX_LEN = 20  # Max sequence length for padding

        self.vocab = vocab

        # print(self.texts)
        # print(self.labels)
        # print(self.vocab)

    def text_to_tensor(self, text, max_len=50):
        # Convert text to indices
        tokens = text.split()
        indices = [self.vocab[token] for token in tokens]
        
        # Pad or truncate to fixed length
        if len(indices) < max_len:
            indices += [self.vocab['<pad>']] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        return torch.tensor(indices, dtype=torch.long)

    def setup(self, stage=None):
        # Combine texts and labels, then split
        dataset = list(zip(self.texts, self.labels))
        
        # 80% Train, 10% Val, 10% Test
        train_size = int(0.8 * len(dataset))
        val_size = int(0.1 * len(dataset))
        test_size = len(dataset) - train_size - val_size
        
        generator = torch.Generator().manual_seed(42)
        train_set, val_set, test_set = data.random_split(
            dataset, [train_size, val_size, test_size], generator=generator
        )
        
        self.train_data = train_set
        self.val_data = val_set
        self.test_data = test_set

    def train_dataloader(self):
        return data.DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return data.DataLoader(self.val_data, batch_size=self.batch_size)

    def test_dataloader(self):
        return data.DataLoader(self.test_data, batch_size=self.batch_size)


In [29]:
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule

class SpamClassifier(LightningModule):
    def __init__(self, vocab_size, embed_dim=50, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=1)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 1) # Binary classification
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, x):
        x = self.embedding(x)
        _, h_n = self.rnn(x)
        return self.classifier(h_n.squeeze(0)).squeeze(1)

    def training_step(self, batch, batch_idx):
        texts, labels = batch
        logits = self(texts)
        loss = self.loss_fn(logits, labels.float())
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        texts, labels = batch
        logits = self(texts)
        loss = self.loss_fn(logits, labels.float())
        self.log('val_loss', loss)
        return loss

    def test_step(self, batch, batch_idx):
        texts, labels = batch
        logits = self(texts)
        loss = self.loss_fn(logits, labels.float())
        self.log('test_loss', loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


In [30]:
from pytorch_lightning import Trainer

# # # Mock Data
# # dummy_texts = ["free money click here", "hey can we meet up?", "you won a prize", "call me back"]
# # dummy_labels = [1, 0, 1, 0] # 1 = Spam, 0 = Ham

# Initialize Data
dm = SMSDataModule(data=df)
dm.build_vocab()
dm.setup()

# Initialize Model
model = SpamClassifier(vocab_size=len(dm.vocab))

# Train and Test
trainer = Trainer(max_epochs=5)
trainer.fit(model, datamodule=dm)
trainer.test(model, datamodule=dm)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/carl/.virtualenvs/mlops-exer-1/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoi

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding         │  473 K │ train │     0 │
│ 1 │ rnn        │ GRU               │ 22.3 K │ train │     0 │
│ 2 │ classifier │ Linear            │     65 │ train │     0 │
│ 3 │ loss_fn    │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 496 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 496 K                                                                                                
Total estimated model params size (MB): 1.985                                                                      
Modules in train mode: 4                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/carl/.virtualenvs/mlops-exer-1/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/carl/.virtualenvs/mlops-exer-1/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connect
or.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value
of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not tuple